# Create the LCIs file for use with *premise*

This notebook imports the raw individual LCIs for lithium projects (update versions of those LCIs provided in Schenker & Pfister (2025)) and export them into a unique Excel file that can be used with premise. This is done for the baseline LCIs (average technology parameters) and the optimized LCIs (optimistic technology parameters)

Some modifications are done on the lithium LCIs:

- Update some metadata of activities like "flow" to "reference product", "amount" to "production amount", add "database", etc.
- Some process names are used across projects (e.g., df_DLE_evaporation_ponds) - problematic to have everything in the same database - create unique activity names by adding the project name
- Change electricity from high voltage to medium voltage and update electricity and heat source to project-specific
- Generate unique code with wurst
- Location of foreground dataset is the name of the project (e.g., Chaerhan) - change to ISO location
- Reference product is missing in activities and exchanges
- Link water flows to the new biosphere database with regionalized water flows
- Add "input" field for foreground processes
- Inputs of "heat production, natural gas, at industrial furnace >100kW" and "machine operation, diesel, >= 74.57 kW, high load factor" have same location as project due to regionalization - change location to RoW
- Ensure biosphere flows have positive sign and waste treatments negative

Moreover, LCIs are created for industrial electric boiler and heat pumps for each project

In [1]:
import bw2data as bd
import bw2io as bi
import wurst
import pandas as pd
from pathlib import Path
import shutil
import copy
from utils import (
    get_ds_for_location, 
    create_electric_boiler_activity,
    create_heat_pump_activity)

10:52:03+0200 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


c:\Users\istrateir\AppData\Local\miniconda3\envs\plca\Lib\site-packages\bw2calc\__init__.py:57: UserWarning: No fast sparse solver found
  warnings.warn("No fast sparse solver found")


In [2]:
BW_PROJECT = 'plca_lithium'
bd.projects.set_current(BW_PROJECT)

LITHIUM_DB = "lithium_brine_projects"
ECOINVENT_DB = "ecoinvent-3.10.1-cutoff"
BIOSPHERE_DB = "ecoinvent-3.10.1-biosphere"
WATER_DB = "biosphere-3.10.1-water regionalized"

technosphere = lambda x: x["type"] == "technosphere"
biosphere = lambda x: x["type"] == "biosphere"
production = lambda x: x["type"] == "production"
economic_flows = lambda x: x["type"] in ["technosphere", "production"]

## Import raw lithium LCIs

In [4]:
def import_lci_dataset(LCI_PATH, ei_name, bio_name):
    """
    return dictionary containing the LCI: key: name of the project \ value
    """
    lci_files = [file for file in LCI_PATH.glob("*.xlsx")] + [file for file in LCI_PATH.glob("*.xls")]
    lci_dict = {}
    for file in lci_files:
        lci = bi.ExcelImporter(file)
        lci.apply_strategies(verbose=False)
        lci.match_database(ei_name, fields=('name', 'reference product', 'unit', 'location'))
        lci.match_database(bio_name, fields=('name', 'unit', 'categories'))
        lci_dict.update({lci.db_name: copy.deepcopy(lci.data)})
    return lci_dict

lci_baseline_dict = import_lci_dataset(Path("../inventories/baseline"), ECOINVENT_DB, BIOSPHERE_DB)
lci_optimized_dict = import_lci_dataset(Path("../inventories/optimized"), ECOINVENT_DB, "ecoinvent-3.10.1-biosphere")

Extracted 1 worksheets in 0.00 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Extracted 1 worksheets in 0.02 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Extracted 1 worksheets in 0.01 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Extracted 1 worksheets in 0.01 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Extracted 1 worksheets in 0.01 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Extracted 1 worksheets in 0.01 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Extracted 1 worksheets in 0.01 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Extracted 1 worksheets in 0.02 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields


## Formating LCI datasets metadata

In [5]:
# Import lithium project information
lithium_projects = pd.read_excel(Path("../scenario_data/lithium_projects_list.xlsx"))
project_locations = dict(zip(lithium_projects["Project name"], lithium_projects["Location"]))
print(len(project_locations))
print(project_locations)

24
{'Salar de Cauchari-Olaroz': 'AR', 'Chaerhan': 'CN-NWG', 'Salar de Centenario': 'AR', 'East Taijinar': 'CN-NWG', 'Lakkor Tso': 'CN-NWG', 'Qinghai Yiliping': 'CN-NWG', 'Salar de Atacama': 'CL', 'Salar de Olaroz': 'AR', 'Fenix': 'AR', 'Silver Peak': 'US-WECC', 'Tres Quebradas': 'AR', 'Zhabuye': 'CN-NWG', 'Sal de los Angeles': 'AR', 'Salar del Hombre Muerto North': 'AR', 'Kachi': 'AR', 'Maricunga': 'CL', 'Salar de Pastos Grandes': 'AR', 'Pozuelos': 'AR', 'Sal de Vida': 'AR', 'Salar de Arizaro': 'AR', 'Salar de Tolillar': 'AR', 'Salar del Rincon': 'AR', 'Salar de Uyuni': 'BO', 'Upper Rhine Graben': 'DE'}


In [6]:
def format_lci_ds(lcis_dict_raw):
    # First drop projects that are not within the scenarios
    lcis_dict = {}
    for project in lcis_dict_raw:
        if project in project_locations.keys():
            lcis_dict[project] = lcis_dict_raw[project]

    # Format the remaining projects
    for project in lcis_dict:
        # Create unique names for datasets associated with the project name
        unique_names = {ds["name"]: ds["name"] + "_" + project for ds in lcis_dict[project]}

        # Some changes in datasets metadata
        for ds in lcis_dict[project]:
            if "flow" in ds:
                ds['reference product'] = ds.pop('flow')
            if "amount" in ds:
                ds['production amount'] = ds.pop('amount')
            ds["database"] = LITHIUM_DB

            # Change to unique activity name
            if ds["name"] in unique_names.keys():
                ds.update({"name": unique_names[ds["name"]]})

            # Generate unique code
            ds.update({"code": wurst.filesystem.get_uuid()})

            # Change location of activity
            if ds["location"] in project_locations.keys():
                ds.update({"location": project_locations[ds["location"]]})

        # Make some changes in the exchange flows:
        for ds in lcis_dict[project]:
            for exc in filter(economic_flows, ds["exchanges"]):

                # Add product and input to production exchanges:
                # "product" is used instead of "reference product" for wurst
                if exc in filter(production, ds["exchanges"]):
                    exc.update({
                        "product": ds["reference product"],
                        "input": (ds["database"], ds["code"])})

                # Change exchange name to unique name
                if exc["name"] in unique_names.keys():
                    exc.update({"name": unique_names[exc["name"]]})
                    
                if exc["location"] in project_locations.keys():
                    exc.update({"location": project_locations[exc["location"]]})

                # Change from high to medium voltage
                if "electricity, high voltage" in exc["name"]:
                    exc.update({"name": "market for electricity, medium voltage"})
                    exc["product"] = "electricity, medium voltage"

                # Waste liquid input positive:
                if "waste_liquid" in exc["name"]:
                    exc["amount"] = abs(exc["amount"])

                # All input treatments have negative amounts:
                if "treatment of" in exc["name"] or "hazardous waste" in exc["name"] or "wastewater" in exc["name"]:
                    exc["amount"] = - abs(exc["amount"])
            
            # All biosphere flows must be negative
            for exc in filter(biosphere, ds["exchanges"]):
                exc["amount"] = abs(exc["amount"])

    return lcis_dict

lci_baseline_dict = format_lci_ds(lci_baseline_dict)
lci_optimized_dict = format_lci_ds(lci_optimized_dict)

### Find missing background and biosphere flows
Due to regionalization in the raw lithium LCIs

In [7]:
# Find activities in the lithium LCIs that are not in ecoinvent
def find_no_ecoinvent_lcis(lcis_dict):
    # Find if there is any background dataset that is not available in ecoinvent
    list_of_foreground_ds = [ds["name"] for project in lcis_dict for ds in lcis_dict[project]]
    list_of_ei_ds = [(ds["name"], ds["location"]) for ds in bd.Database(ECOINVENT_DB)]

    no_ecoinvent_lcis = []
    for project in lcis_dict:
        for ds in lcis_dict[project]:
            for exc in filter(technosphere, ds["exchanges"]):
                # Do this only for background datasets; i.e., datasets that are not in the project associated datasets
                if exc["name"] not in list_of_foreground_ds:
                    exc_ds = [ds for ds in list_of_ei_ds if ds[0] == exc["name"] and ds[1] == exc["location"]]
                    if len(exc_ds) == 0:
                        no_ecoinvent_lcis.append((exc["name"], exc["location"]))

    no_ecoinvent_lcis = list(set(no_ecoinvent_lcis))
    return no_ecoinvent_lcis

no_ecoinvent_lcis_baseline = find_no_ecoinvent_lcis(lci_baseline_dict)
no_ecoinvent_lcis_optimized = find_no_ecoinvent_lcis(lci_optimized_dict)

In [8]:
no_ecoinvent_lcis_baseline

[('heat production, natural gas, at industrial furnace >100kW', 'AR'),
 ('market for wastewater, average', 'US-WECC'),
 ('machine operation, diesel, >= 74.57 kW, high load factor', 'CN-NWG'),
 ('heat production, natural gas, at industrial furnace >100kW', 'DE'),
 ('heat production, natural gas, at industrial furnace >100kW', 'CL'),
 ('machine operation, diesel, >= 74.57 kW, high load factor', 'AR'),
 ('heat production, natural gas, at industrial furnace >100kW', 'BO'),
 ('market for wastewater, average', 'CN-NWG'),
 ('market for wastewater, average', 'AR'),
 ('machine operation, diesel, >= 74.57 kW, high load factor', 'CL'),
 ('heat production, natural gas, at industrial furnace >100kW', 'US-WECC'),
 ('machine operation, diesel, >= 74.57 kW, high load factor', 'BO'),
 ('deep well drilling, for deep geothermal power reg', 'DE'),
 ('market for wastewater, average', 'DE'),
 ('sodium hydroxide to generic market for neutralising agent', 'GLO'),
 ('market for soda ash, light', 'GLO'),
 ('mar

In [9]:
no_ecoinvent_lcis_optimized

[('heat production, natural gas, at industrial furnace >100kW', 'AR'),
 ('market for wastewater, average', 'US-WECC'),
 ('machine operation, diesel, >= 74.57 kW, high load factor', 'CN-NWG'),
 ('heat production, natural gas, at industrial furnace >100kW', 'DE'),
 ('heat production, natural gas, at industrial furnace >100kW', 'CL'),
 ('machine operation, diesel, >= 74.57 kW, high load factor', 'AR'),
 ('heat production, natural gas, at industrial furnace >100kW', 'BO'),
 ('market for wastewater, average', 'CN-NWG'),
 ('market for wastewater, average', 'AR'),
 ('machine operation, diesel, >= 74.57 kW, high load factor', 'CL'),
 ('heat production, natural gas, at industrial furnace >100kW', 'US-WECC'),
 ('machine operation, diesel, >= 74.57 kW, high load factor', 'BO'),
 ('deep well drilling, for deep geothermal power reg', 'DE'),
 ('market for wastewater, average', 'DE'),
 ('sodium hydroxide to generic market for neutralising agent', 'GLO'),
 ('market for soda ash, light', 'GLO'),
 ('mar

In [10]:
# Modify the missing background inputs to others that exist in ecoinvent
def update_background_inputs(lcis_dict):
    # Flatten all foreground datasets once
    foreground_lookup = {d["name"]: d for project in lcis_dict for d in lcis_dict[project]}

    # Build a fast lookup for the background database once
    background_lookup = {(ds["name"], ds["location"], ds["unit"]): ds for ds in bd.Database(ECOINVENT_DB)}

    for project in lcis_dict:
        for ds in lcis_dict[project]:
            # Add product and input to technosphere exchanges:
            for exc in filter(technosphere, ds["exchanges"]):

                # Find datasets for foreground exchanges
                if exc["name"] in foreground_lookup:
                    try:
                        exc_ds = [foreground_lookup[exc["name"]]]
                    except KeyError:
                        exc_ds = []
                else:
                    # apply location/name adjustments first
                    special_location_map = {
                        "heat production, natural gas, at industrial furnace >100kW": "RoW",
                        "market for wastewater, average": "RoW",
                        "market for soda ash, light": "RoW",
                    }
                    if exc["name"] in special_location_map:
                        exc.update({"location": special_location_map[exc["name"]]})

                    if exc["name"] == "machine operation, diesel, >= 74.57 kW, high load factor":
                        exc.update({"location": "GLO"})

                    if exc["name"] == "deep well drilling, for deep geothermal power reg":
                        exc.update({"name": "deep well drilling, for deep geothermal power"})
                        if exc["location"] == "US-WECC":
                            exc.update({"location": "US-HICC"})

                    # Change sodium hydroxide generic market to market for sodium hydroxide
                    if exc["name"] == "sodium hydroxide to generic market for neutralising agent":
                        exc.update({
                            "name": "market for sodium hydroxide, without water, in 50% solution state",
                            "product": "sodium hydroxide, without water, in 50% solution state",
                            "location": "RoW"})

                    exc_ds = [background_lookup.get((exc["name"], exc["location"], exc["unit"]))]
                    exc_ds = [ds for ds in exc_ds if ds]  # drop None

                if len(exc_ds) > 1:
                    raise ValueError("More than one dataset for exchange", (exc["name"], exc["location"])) 
                            
                if len(exc_ds) == 0:
                    raise ValueError("LCI dataset not found for", (exc["name"], exc["location"]))
                        
                # "product" is used instead of "reference product" for wurst
                exc.update({
                    "product": exc_ds[0]["reference product"],
                    "input": (exc_ds[0]["database"], exc_ds[0]["code"])})
    return lcis_dict

lci_baseline_dict = update_background_inputs(lci_baseline_dict)
lci_optimized_dict = update_background_inputs(lci_optimized_dict)

In [11]:
def find_no_biosphere_flows(lci_dicts):
    # Find biosphere flows that are not in the biosphere database
    list_of_bio_ds = [(ds["name"], ds["categories"]) for ds in bd.Database(BIOSPHERE_DB)]
    no_biosphere_flows = []
    for project in lci_dicts:
        for ds in lci_dicts[project]:
            for exc in filter(biosphere, ds["exchanges"]):
                if (exc["name"], exc["categories"])  not in list_of_bio_ds:
                    no_biosphere_flows.append(exc["name"])
    return no_biosphere_flows

no_biosphere_flow_baseline = find_no_biosphere_flows(lci_baseline_dict)
no_biosphere_flow_optimized = find_no_biosphere_flows(lci_optimized_dict)

In [12]:
print(list(set(no_biosphere_flow_baseline)))
print(list(set(no_biosphere_flow_optimized)))

['Sodium']
['Sodium']


In [13]:
def update_biosphere_flows(lcis_dict):
    # Update "Sodium" to "Sodium I"
    sodium_i_ds = [ds for ds in bd.Database(BIOSPHERE_DB) if ds["name"]=="Sodium I" and ds["categories"]==("water",)][0]

    for project in lcis_dict:
        for ds in lcis_dict[project]:
            for exc in filter(biosphere, ds["exchanges"]):
                if exc["name"] == "Sodium":
                    exc["name"] = "Sodium I"
                    exc["input"] = sodium_i_ds.key
    return lcis_dict

lci_baseline_dict = update_biosphere_flows(lci_baseline_dict)
lci_optimized_dict = update_biosphere_flows(lci_optimized_dict)

## Update electricity and heat

The simulation of the optimized LCIs assumed a reduction in heat losses from 15% to 5%. However,
this may be overly optimistic for lithium projects operating at high altitude where reduced oxygen partial 
pressure can affect combustion efficiency. Therefore, the heat consumption in the optimized LCIs is increased by
10% to reflect the 15% heat losses (therefore, no heat losses reduction in the optimized LCIs)

This code also creates new LCIs for industrial electric boilers and heat pumps for each project
considering the project-specific electricity source

**Adjust heat losses**

In [20]:
for project in lci_optimized_dict:
    for ds in lci_optimized_dict[project]:
        for ex in ds["exchanges"]:
            if "heat production" in ex["name"]:
                ex["amount"] = ex["amount"] * 1.1

**Adapt electricity and heat sources**

In [22]:
# By default, all projects use grid electricity and natural gas heat

# Map electricity/heat sources to ecoinvent activities (name, reference product, location)
# Industrial electric boiler is not included in ecoinvent, so it will be created here

ELEC_SOURCE_MAPPING = {
    "electricity_grid": ("market for electricity, medium voltage", "electricity, medium voltage"),
    "natural_gas": ("electricity production, natural gas, conventional power plant", "electricity, high voltage"),
    "diesel": ("diesel, burned in diesel-electric generating set, 10MW", "diesel, burned in diesel-electric generating set, 10MW"),
    "natural_gas_CHP": ("heat and power co-generation, natural gas, conventional power plant, 100MW electrical", "electricity, high voltage"),
    "diesel_CHP": ("heat and power co-generation, diesel, 200kW electrical, SCR-NOx reduction", "electricity, high voltage"),
}

HEAT_SOURCE_MAPPING = {
    "diesel": ("heat production, light fuel oil, at industrial furnace 1MW", "heat, district or industrial, other than natural gas"),
    "propane": ("heat production, propane, at industrial furnace >100kW", "heat, district or industrial, other than natural gas"),
    "natural_gas_CHP": ("heat and power co-generation, natural gas, conventional power plant, 100MW electrical", "heat, district or industrial, natural gas"),
    "diesel_CHP": ("heat and power co-generation, diesel, 200kW electrical, SCR-NOx reduction", "heat, district or industrial, other than natural gas"),
}

In [23]:
# Create an LCI for industrial electric boiler and heat pump for each project location
# using the project-specific electricity source
electrified_heat_lcis = []

for project in project_locations:

    # Get electricity source
    proj_elec_supply = lithium_projects[lithium_projects["Project name"]==project]["Electricity supply"].values[0]
    elec_supply_ds = get_ds_for_location(
                ELEC_SOURCE_MAPPING[proj_elec_supply][0], 
                ELEC_SOURCE_MAPPING[proj_elec_supply][1],
                project_locations[project], ECOINVENT_DB)[0]
    
    # Create corresponding electric boiler activity
    boiler_actv_name = f"heat production, industrial electric boiler, {project}"
    boiler_lci = create_electric_boiler_activity(boiler_actv_name, project_locations[project], LITHIUM_DB, elec_supply_ds)
    electrified_heat_lcis.append(boiler_lci)

    # Create corresponding heat pump activity
    heat_pump_actv_name = f"heat production, at heat pump 30kW, allocation exergy, {project}"
    heat_pump_lci = create_heat_pump_activity(heat_pump_actv_name, project_locations[project], LITHIUM_DB, elec_supply_ds, ECOINVENT_DB)
    electrified_heat_lcis.append(heat_pump_lci)

In [24]:
def update_energy_source(lcis_dict):
    """"
    This function updates the lithium LCIs to match the project-specific electricity and heat source

    Diesel boiler efficiency: 83% (Schoeneberger et al 2022 - Advances in Applied Energy 5, 100089)
    """

    DIESEL_GENERATOR_EFFICIENCY = 0.4

    for project in lcis_dict:
        # Get the electricity and heat source
        proj_elec_source = lithium_projects[lithium_projects["Project name"]==project]["Electricity supply"].values[0]
        proj_heat_source = lithium_projects[lithium_projects["Project name"]==project]["Heat supply"].values[0]

        # Adjust electricity source if different from grid
        if proj_elec_source != "electricity_grid":
            print(f"Adjusting electricity source for project {project} in location {project_locations[project]}. New electricity source: {proj_elec_source}")

            # Find electricity supply dataset for project
            elec_supply_ds = get_ds_for_location(
                ELEC_SOURCE_MAPPING[proj_elec_source][0], 
                ELEC_SOURCE_MAPPING[proj_elec_source][1],
                project_locations[project], ECOINVENT_DB)[0]
            print(f"Electricity source activity: {elec_supply_ds['name']}, {elec_supply_ds['reference product']}, {elec_supply_ds['location']}")

            for ds in lcis_dict[project]:
                for exc in filter(technosphere, ds["exchanges"]):
                    if "market for electricity" in exc["name"]:
                        electricity_amount = exc["amount"]

                        # Diesel generator has MJ diesel burned as unit.
                        # Convert kWh electricity to MJ diesel assuming the efficiency
                        if proj_elec_source == "diesel":
                            electricity_amount = exc["amount"] * 3.6 / DIESEL_GENERATOR_EFFICIENCY
                        
                        exc.update({
                        'name': elec_supply_ds['name'],
                        'product': elec_supply_ds['reference product'],
                        "amount": electricity_amount,
                        'unit': elec_supply_ds['unit'],
                        'location': elec_supply_ds['location'],
                        'input': (elec_supply_ds['database'], elec_supply_ds['code'])
                        })

        # Adjust heat source if other than natural gas
        if proj_heat_source != "natural_gas":
            print(f"Adjusting heat source for project {project} in location {project_locations[project]}. New heat source: {proj_heat_source}")

            # Find heat supply dataset for project location
            if proj_heat_source == "electric_boiler":
                heat_supply_ds = [ds for ds in electrified_heat_lcis 
                    if project in ds["name"] and "electric boiler" in ds["name"]][0]
            else:
                heat_supply_ds = get_ds_for_location(
                    HEAT_SOURCE_MAPPING[proj_heat_source][0], 
                    HEAT_SOURCE_MAPPING[proj_heat_source][1],
                    project_locations[project], ECOINVENT_DB)[0]
            print(f"Heat source activity: {heat_supply_ds['name']}, {heat_supply_ds['reference product']}, {heat_supply_ds['location']}")

            for ds in lcis_dict[project]:
                for exc in filter(technosphere, ds["exchanges"]):
                    if "heat production" in exc["name"]:

                        exc.update({
                          'name': heat_supply_ds['name'],
                          'product': heat_supply_ds['reference product'],
                          'unit': heat_supply_ds['unit'],
                          'location': heat_supply_ds['location'],
                          'input': (heat_supply_ds['database'], heat_supply_ds['code'])
                        })
    
    return lcis_dict

lci_baseline_dict = update_energy_source(lci_baseline_dict)
lci_optimized_dict = update_energy_source(lci_optimized_dict)

Adjusting heat source for project Maricunga in location CL. New heat source: diesel
Heat source activity: heat production, light fuel oil, at industrial furnace 1MW, heat, district or industrial, other than natural gas, RoW
Adjusting electricity source for project Pozuelos in location AR. New electricity source: natural_gas_CHP
Electricity source activity: heat and power co-generation, natural gas, conventional power plant, 100MW electrical, electricity, high voltage, AR
Adjusting heat source for project Pozuelos in location AR. New heat source: natural_gas_CHP
Heat source activity: heat and power co-generation, natural gas, conventional power plant, 100MW electrical, heat, district or industrial, natural gas, AR
Adjusting electricity source for project Sal de Vida in location AR. New electricity source: diesel
Electricity source activity: diesel, burned in diesel-electric generating set, 10MW, diesel, burned in diesel-electric generating set, 10MW, GLO
Adjusting heat source for projec

## Relink biosphere flows and water flows to regionalized biosphere

In [25]:
def relink_biosphere_flows(lci_dicts):
    water_flows_regionalized = [act for act in bd.Database(WATER_DB)]

    for project in lci_dicts:
        for ds in lci_dicts[project]:
            for ex in ds["exchanges"]:
                if ex["type"]=="biosphere":

                    # First check if the biosphere flow is in the regionalized database
                    bio_flow = [flow for flow in water_flows_regionalized 
                                if flow["name"] == ex["name"] 
                                and flow["categories"] == ex["categories"] 
                                and flow["location"] == project]
                    if len(bio_flow)==0:
                        bio_flow = [flow for flow in bd.Database(BIOSPHERE_DB) 
                                    if flow["name"] == ex["name"] 
                                    and flow["categories"] == ex["categories"]]                 
                        if len(bio_flow) == 0:
                            raise ValueError(f"No biosphere exchange found for flow {ex['name']}")
                    
                    ex.update({"input": (bio_flow[0]["database"], bio_flow[0]["code"])})
    return lci_dicts

lci_baseline_dict = relink_biosphere_flows(lci_baseline_dict)
lci_optimized_dict = relink_biosphere_flows(lci_optimized_dict)

## Export LCIs

In [26]:
lci_all_baseline = []
for project in lci_baseline_dict: 
    lci_all_baseline.extend(copy.deepcopy(lci_baseline_dict[project]))
lci_all_baseline.extend(copy.deepcopy(electrified_heat_lcis))

lci_all_optimized = []
for project in lci_optimized_dict:
    lci_all_optimized.extend(copy.deepcopy(lci_optimized_dict[project]))
lci_all_optimized.extend(copy.deepcopy(electrified_heat_lcis))

DB_OPTIMIZED_NAME = LITHIUM_DB + "-optimized"
for ds in lci_all_optimized:
    if ds["database"] == 'lithium_brine_projects':
        ds["database"] = DB_OPTIMIZED_NAME
    for ex in ds["exchanges"]:
        if ex["input"][0] == 'lithium_brine_projects':
            ex["input"] = (DB_OPTIMIZED_NAME, ex["input"][1])

In [27]:
if LITHIUM_DB in bd.databases:
    del bd.databases[LITHIUM_DB]
db_baseline = bd.Database(LITHIUM_DB)
db_baseline.write(lci_all_baseline)

11:07:53+0200 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|██████████| 480/480 [00:00<00:00, 1685.83it/s]


11:08:01+0200 [info     ] Vacuuming database            


In [28]:
if DB_OPTIMIZED_NAME in bd.databases:
    del bd.databases[DB_OPTIMIZED_NAME]
db_opt = bd.Database(DB_OPTIMIZED_NAME)
db_opt.write(lci_all_optimized)

11:09:26+0200 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|██████████| 480/480 [00:00<00:00, 984.27it/s] 


11:09:36+0200 [info     ] Vacuuming database            


In [29]:
# Export inventories to Excel file
for db in [LITHIUM_DB, DB_OPTIMIZED_NAME]:
    export_path = bi.export.excel.write_lci_excel(db)
    filepath = Path("../inventories")
    shutil.copy(export_path, filepath)